# Algorithm Spot-Checking

This notebook performs spot-checking of different classification algorithms using a reusable pipeline built with `imblearn` and `scikit-learn`.

**Data flow:**
- Input: `kidney_disease_encoded.csv` — dataset with all features already encoded (generated by `Encode_Categorical_Values.ipynb`)
- Normalization is applied **inside the pipeline**, ensuring that `MinMaxScaler` is fitted only on the training data of each fold (no data leakage)
- SMOTE balancing is applied only to the training data of each fold

**Approach:**
- Stratified cross-validation (`StratifiedKFold`, k=5)
- Metrics: F1, Accuracy, Precision and Recall
- Classifier and/or sampler can be swapped without changing the pipeline structure

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline as SklearnPipeline

# Classificadores
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# Pipeline com suporte a samplers (imblearn)
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

## 2. Load and prepare data

In [ ]:
# Load encoded dataset
df = pd.read_csv("../data/processed/kidney_disease_encoded.csv")

print(f"Shape: {df.shape}")
print(f"\nTarget variable distribution:")
print(df["class"].value_counts())

In [ ]:
X = df.drop(columns=["class"])
y = LabelEncoder().fit_transform(df["class"])  # ckd=0, not ckd=1 (or vice versa)

print(f"Features: {X.shape[1]} columns, {X.shape[0]} samples")
print(f"Encoded classes: {np.unique(y)} (original: {df['class'].unique()})")

## 3. Reusable Pipeline

The `make_pipeline` function encapsulates:
1. **`MinMaxScaler`** — scaling fitted only on the training data of each fold  
2. **Sampler (SMOTE by default)** — balancing applied only to the training data  
3. **Classifier** — interchangeable without changing the rest of the pipeline  

> `imblearn.pipeline.Pipeline` is used instead of `sklearn.pipeline.Pipeline` to support samplers (which change the number of samples via `fit_resample`).

In [ ]:
def make_pipeline(classifier, sampler=SMOTE(random_state=42)):
    steps = [
        ("scaler", MinMaxScaler()),
        ("classifier", classifier),
    ]
    if sampler is not None:
        steps.insert(1, ("sampler", sampler))
    return Pipeline(steps)

## 4. Algorithms Definition

In [ ]:
models = {
    "KNN":      KNeighborsClassifier(),
    "DTree":    DecisionTreeClassifier(random_state=42),
    "RF":       RandomForestClassifier(random_state=42),
    "SVM":      SVC(random_state=42),
    "NB":       GaussianNB(),
}

## 5. Spot-Checking with K-Fold Cross Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# cv_df will now store the accuracy scores from all folds for each model (models as columns, folds as rows)
cv_f1_per_model_folds = {}

# cv_df_mean will store the mean and std dev of all metrics for each model
cv_mean_std_data = {}

scoring = ['accuracy', 'precision', 'recall', 'f1']

for name, model_candidate in models.items():
    model = model_candidate
    if isinstance(model_candidate, type): # Check if it's a class (e.g., KNeighborsClassifier)
        model = model_candidate() # Instantiate the class

    pipeline = make_pipeline(model)

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )

    # Store f1 scores for cv_df (for boxplot compatibility)
    cv_f1_per_model_folds[name] = scores['test_f1'].tolist()

    # Calculate mean and std for cv_df_mean
    model_stats = {}
    for metric in scoring:
        model_stats[f'{metric}_mean'] = np.mean(scores[f'test_{metric}'])
        model_stats[f'{metric}_std'] = np.std(scores[f'test_{metric}'])
    cv_mean_std_data[name] = model_stats

# Create cv_df
cv_df = pd.DataFrame(cv_f1_per_model_folds)

# Create cv_df_mean (mean and std dev for all metrics)
cv_df_mean = pd.DataFrame(cv_mean_std_data).T.sort_values(by="f1_mean", ascending=False)

display(cv_df_mean)


## 6. Boxplot (CV)

In [ ]:
cv_df.boxplot()
plt.title("Performance Distribution (CV)")
plt.ylabel("F1-score")
plt.show()